In [1]:
import re
import os
import json
import pm4py
import pandas as pd
from collections import Counter
from IPython.display import display
from pm4py.util import constants
from pm4py.objects.log.obj import EventLog

from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer

LOG_DIR             = "role_logs"
FULL_LOG_PATH       = "commitizen_role_annotated.json"
FLATTEN_OBJECT_TYPE = "issue"   # each issue becomes a case in the traditional log
MIN_SUPPORT         = 0.5       # fraction of cases a constraint must hold
ITEMSETS_SUPPORT    = 0.9       # support threshold for frequent-itemset pre-filter
MAX_CARDINALITY     = 2         # max cardinality for DECLARE templates (e.g. Existence2)
CONSIDER_VACUITY    = False     # vacuously satisfied constraints count as violated

# Roles with more than one distinct activity — the only ones suitable for
# per-role DECLARE discovery when flattened by 'issue'.
DECLARE_ROLES = {"bot", "issue_reporter"}

role_files = sorted([
    f for f in os.listdir(LOG_DIR)
    if f.startswith("ocel_") and f.endswith(".json")
])

print("Configuration:")
print(f"  Flatten object type : {FLATTEN_OBJECT_TYPE}")
print(f"  Min support         : {MIN_SUPPORT}")
print(f"  Itemsets support    : {ITEMSETS_SUPPORT}")
print(f"  Max cardinality     : {MAX_CARDINALITY}")
print(f"  Consider vacuity    : {CONSIDER_VACUITY}")
print(f"  Role logs found     : {len(role_files)}")

Configuration:
  Flatten object type : issue
  Min support         : 0.5
  Itemsets support    : 0.9
  Max cardinality     : 2
  Consider vacuity    : False
  Role logs found     : 8


In [2]:
# ── Helper: OCEL path → D4PyEventLog ─────────────────────────────────────────
#
# Declare4Py works on traditional (case-centric) event logs.
# We flatten each OCEL log on FLATTEN_OBJECT_TYPE so that each object of that
# type becomes a case whose trace is all events touching that object.

def ocel_to_d4py(path: str, object_type: str = FLATTEN_OBJECT_TYPE) -> tuple[D4PyEventLog, pd.DataFrame]:
    """Read an OCEL2 JSON, flatten on object_type, return (D4PyEventLog, flat_df)."""
    ocel    = pm4py.read_ocel2_json(path)
    flat_df = pm4py.ocel_flattening(ocel, object_type)
    el: EventLog = pm4py.convert_to_event_log(flat_df)
    # Declare4Py requires these two properties to be present in the EventLog
    el._properties[constants.PARAMETER_CONSTANT_ACTIVITY_KEY]  = "concept:name"
    el._properties[constants.PARAMETER_CONSTANT_TIMESTAMP_KEY] = "time:timestamp"
    return D4PyEventLog(log=el), flat_df


def parse_constraint(raw: str) -> dict:
    """Parse a Declare4Py constraint string like 'Response[A, B] | |' into components."""
    m = re.match(r'([^\[]+)\[([^\]]+)\]', raw.strip())
    if not m:
        return {"template": raw, "activity_a": "", "activity_b": ""}
    template   = m.group(1).strip()
    activities = [a.strip() for a in m.group(2).split(",")]
    return {
        "template":   template,
        "activity_a": activities[0] if len(activities) > 0 else "",
        "activity_b": activities[1] if len(activities) > 1 else "",
    }


# Load full log and all role logs
print("Loading and flattening logs ...")
d4py_full, flat_full = ocel_to_d4py(FULL_LOG_PATH)

d4py_roles: dict[str, D4PyEventLog] = {}
flat_roles: dict[str, pd.DataFrame] = {}
for fname in role_files:
    role = fname.replace("ocel_", "").replace(".json", "")
    d4py_roles[role], flat_roles[role] = ocel_to_d4py(os.path.join(LOG_DIR, fname))

# Summary table
rows = [{"source": "full log",
         "cases": flat_full["case:concept:name"].nunique(),
         "events": len(flat_full),
         "distinct_activities": flat_full["concept:name"].nunique()}]
for role, df in sorted(flat_roles.items()):
    rows.append({"source": role,
                 "cases": df["case:concept:name"].nunique(),
                 "events": len(df),
                 "distinct_activities": df["concept:name"].nunique()})

print(f"\nFlattened log summary (object type = '{FLATTEN_OBJECT_TYPE}'):")
display(pd.DataFrame(rows).set_index("source"))

Loading and flattening logs ...


/Users/liorlimonad/miniforge3/lib/python3.10/site-packages/pm4py/util/dt_parsing/parser.py:77: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(



Flattened log summary (object type = 'issue'):


,cases,events,distinct_activities
source,,,
full log,1459,21488,35
bot,1045,3394,11
contributor,109,236,1
devops_engineer,7,15,1
feature_developer,27,100,1
issue_reporter,1436,16037,34
maintainer,837,2149,1
quality_engineer,60,194,1
technical_writer,37,71,1


In [3]:
# ── DECLARE discovery on the full issue log ───────────────────────────────────
#
# DeclareMiner uses the Apriori algorithm to find frequent activity itemsets
# and then checks which DECLARE templates hold above MIN_SUPPORT.

miner_full = DeclareMiner(
    log=d4py_full,
    consider_vacuity=CONSIDER_VACUITY,
    min_support=MIN_SUPPORT,
    itemsets_support=ITEMSETS_SUPPORT,
    max_declare_cardinality=MAX_CARDINALITY,
)
model_full = miner_full.run()

raw_constraints = model_full.get_decl_model_constraints()
print(f"DECLARE model (full log, support≥{MIN_SUPPORT}):")
print(f"  Total constraints discovered: {len(raw_constraints)}")
print()

# Parse into a readable DataFrame
df_constraints = pd.DataFrame([parse_constraint(c) for c in raw_constraints])
df_constraints["raw"] = raw_constraints
display(df_constraints.drop(columns="raw"))

Computing discovery ...
DECLARE model (full log, support≥0.5):
  Total constraints discovered: 21



,template,activity_a,activity_b
0,Existence1,created,
1,Absence2,created,
2,Exactly1,created,
3,Init,created,
4,Existence1,closed,
5,Absence2,closed,
6,Exactly1,closed,
7,Choice,created,closed
8,Choice,closed,created
9,Responded Existence,created,closed


In [4]:
# ── Constraint type breakdown ─────────────────────────────────────────────────

TEMPLATE_LEGEND = {
    "Existence1":           "A must occur at least once",
    "Absence2":             "A must occur at most once",
    "Exactly1":             "A must occur exactly once",
    "Init":                 "A must be the first activity",
    "Choice":               "At least one of A or B must occur",
    "Exclusive Choice":     "Exactly one of A or B must occur",
    "Responded Existence":  "If A occurs, B must also occur (order-free)",
    "Co-Existence":         "A and B must both occur or neither",
    "Response":             "Every A must eventually be followed by B",
    "Alternate Response":   "Every A must be followed by B before the next A",
    "Chain Response":       "A must be immediately followed by B",
    "Precedence":           "B can only occur if A occurred before",
    "Alternate Precedence": "Every B must be preceded by A with no other B in between",
    "Chain Precedence":     "B must be immediately preceded by A",
    "Succession":           "Response + Precedence between A and B",
    "Not Response":         "A must never eventually be followed by B",
    "Not Precedence":       "B must not be preceded by A",
    "Not Chain Response":   "A must not be immediately followed by B",
    "Not Chain Precedence": "B must not be immediately preceded by A",
    "Not Co-Existence":     "A and B must never both occur",
}

df_breakdown = (
    df_constraints
    .groupby("template")
    .size()
    .rename("count")
    .sort_values(ascending=False)
    .to_frame()
)
df_breakdown["meaning"] = df_breakdown.index.map(lambda t: TEMPLATE_LEGEND.get(t, "—"))

print("Constraint template breakdown:")
display(df_breakdown)

Constraint template breakdown:


,count,meaning
template,,
Absence2,2,A must occur at most once
Choice,2,At least one of A or B must occur
Exactly1,2,A must occur exactly once
Existence1,2,A must occur at least once
Not Chain Precedence,2,B must not be immediately preceded by A
Not Chain Response,2,A must not be immediately followed by B
Responded Existence,2,"If A occurs, B must also occur (order-free)"
Alternate Precedence,1,Every B must be preceded by A with no other B ...
Alternate Response,1,Every A must be followed by B before the next A


In [5]:
# ── Conformance checking: full log vs. discovered model ───────────────────────
#
# MPDeclareAnalyzer checks every trace against every constraint.
# get_metric('state') returns a (traces × constraints) DataFrame where
# 1 = satisfied and 0 = violated.
# Trace fitness = fraction of constraints satisfied.

analyzer_full = MPDeclareAnalyzer(
    log=d4py_full,
    declare_model=model_full,
    consider_vacuity=CONSIDER_VACUITY,
)
results_full = analyzer_full.run()

state_df = results_full.get_metric("state")
state_df.index = flat_full["case:concept:name"].unique()
state_df.index.name = "case_id"

fitness = state_df.mean(axis=1)

print("Conformance against full DECLARE model:")
print(f"  Traces checked          : {len(fitness)}")
print(f"  Perfectly fitting (1.0) : {(fitness == 1.0).sum()}")
print(f"  Mean fitness            : {fitness.mean():.4f}")
print(f"  Median fitness          : {fitness.median():.4f}")
print(f"  Min fitness             : {fitness.min():.4f}")
print()

# Fitness distribution
bins   = [0.0, 0.7, 0.8, 0.9, 0.95, 1.01]
labels = ["< 0.70", "0.70–0.80", "0.80–0.90", "0.90–0.95", "0.95–1.00"]
bands  = pd.cut(fitness, bins=bins, labels=labels, include_lowest=True)
print("Fitness distribution:")
display(bands.value_counts().sort_index().rename("traces").to_frame())

# Most violated constraints
viol_df  = results_full.get_metric("num_violations")
viol_sum = viol_df.sum().sort_values(ascending=False)
viol_sum = viol_sum[viol_sum > 0]
if not viol_sum.empty:
    print("\nMost violated constraints:")
    display(
        viol_sum.rename("total_violations")
        .reset_index()
        .rename(columns={"index": "constraint"})
    )
else:
    print("No violations found.")

Conformance against full DECLARE model:
  Traces checked          : 1459
  Perfectly fitting (1.0) : 775
  Mean fitness            : 0.9738
  Median fitness          : 1.0000
  Min fitness             : 0.4286

Fitness distribution:


,traces
< 0.70,6
0.70–0.80,1
0.80–0.90,23
0.90–0.95,0
0.95–1.00,1429



Most violated constraints:


,constraint,total_violations
0,"Alternate Precedence[created, closed] | |",23
1,"Response[created, closed] | |",6
2,"Responded Existence[created, closed] | |",6
3,"Alternate Response[created, closed] | |",6
4,"Not Chain Precedence[created, closed] | |",4
5,"Not Chain Response[created, closed] | |",4


In [6]:
# ── Per-role DECLARE discovery ────────────────────────────────────────────────
#
# Roles with only one distinct activity (committed) cannot produce meaningful
# DECLARE models. Only DECLARE_ROLES = {bot, issue_reporter} have enough
# activity diversity when flattened by 'issue'.

declare_role_models: dict[str, object] = {}

for role in sorted(DECLARE_ROLES):
    flat = flat_roles[role]
    n_acts = flat["concept:name"].nunique()

    if n_acts < 2:
        print(f"  {role}: {n_acts} activity — skipping")
        continue

    miner = DeclareMiner(
        log=d4py_roles[role],
        consider_vacuity=CONSIDER_VACUITY,
        min_support=MIN_SUPPORT,
        itemsets_support=ITEMSETS_SUPPORT,
        max_declare_cardinality=MAX_CARDINALITY,
    )
    model = miner.run()
    declare_role_models[role] = model

    raw = model.get_decl_model_constraints()

    print(f"\n{'='*60}")
    print(f"  ROLE: {role.upper()}  "
          f"({flat['case:concept:name'].nunique()} cases, {n_acts} activities)")
    print(f"{'='*60}")
    print(f"  Discovered {len(raw)} constraints")

    if not raw:
        print("  (no constraints above support threshold)")
        continue

    df  = pd.DataFrame([parse_constraint(c) for c in raw])

    print()
    breakdown = df.groupby("template").size().rename("count").sort_values(ascending=False)
    display(breakdown.to_frame())
    print()
    display(df)

Computing discovery ...

  ROLE: BOT  (1045 cases, 11 activities)
  Discovered 0 constraints
  (no constraints above support threshold)
Computing discovery ...

  ROLE: ISSUE_REPORTER  (1436 cases, 34 activities)
  Discovered 3 constraints



,count
template,
Absence2,1
Exactly1,1
Existence1,1


,template,activity_a,activity_b
0,Existence1,closed,
1,Absence2,closed,
2,Exactly1,closed,


In [7]:
# ── Cross-role conformance: each role vs. the full DECLARE model ──────────────
#
# For each role, take its issue-flattened log and measure how well it conforms
# to the constraints derived from the full issue lifecycle.
# Single-activity roles will typically show high fitness because most relational
# constraints are vacuously satisfied (their sole activity is either not the
# antecedent or not the consequent of any binary template).

cross_rows = []
print("Cross-role conformance against full DECLARE model:")
print(f"  ({len(raw_constraints)} constraints, support≥{MIN_SUPPORT})\n")

for role in sorted(flat_roles):
    flat = flat_roles[role]
    if flat.empty:
        cross_rows.append({"role": role, "cases": 0, "mean_fitness": None, "fit_%": None})
        continue

    analyzer = MPDeclareAnalyzer(
        log=d4py_roles[role],
        declare_model=model_full,
        consider_vacuity=CONSIDER_VACUITY,
    )
    rb = analyzer.run()
    s  = rb.get_metric("state")

    trace_fitness   = s.mean(axis=1)
    perfectly_fit   = (trace_fitness == 1.0).sum()
    cross_rows.append({
        "role":         role,
        "cases":        len(s),
        "mean_fitness": round(trace_fitness.mean(), 4),
        "fit_%":        round(100 * perfectly_fit / len(s), 1),
    })

df_cross = (
    pd.DataFrame(cross_rows)
    .set_index("role")
    .sort_values("mean_fitness", ascending=False)
)
display(df_cross)

print()
print("  mean_fitness — avg fraction of constraints satisfied across all issue-traces")
print("  fit_%        — % of issue-traces where every constraint is satisfied")

Cross-role conformance against full DECLARE model:
  (21 constraints, support≥0.5)



,cases,mean_fitness,fit_%
role,,,
issue_reporter,1436,0.8522,73.5
bot,1045,0.2522,4.2
contributor,109,0.0952,0.0
devops_engineer,7,0.0952,0.0
feature_developer,27,0.0952,0.0
maintainer,837,0.0952,0.0
quality_engineer,60,0.0952,0.0
technical_writer,37,0.0952,0.0



  mean_fitness — avg fraction of constraints satisfied across all issue-traces
  fit_%        — % of issue-traces where every constraint is satisfied


In [8]:
# ── Per-role violation breakdown vs. full model ───────────────────────────────
#
# For roles that do show some violations, identify which constraints are
# most frequently broken.

print("Top violated constraints per role (full model):")

for role in sorted(flat_roles):
    flat = flat_roles[role]
    if flat.empty:
        continue

    analyzer = MPDeclareAnalyzer(
        log=d4py_roles[role],
        declare_model=model_full,
        consider_vacuity=CONSIDER_VACUITY,
    )
    rb    = analyzer.run()
    viols = rb.get_metric("num_violations").sum().sort_values(ascending=False)
    viols = viols[viols > 0]

    if viols.empty:
        print(f"\n  {role.upper()}: no violations")
    else:
        print(f"\n  {role.upper()}:")
        for constraint, count in viols.head(5).items():
            parsed = parse_constraint(constraint)
            label  = f"{parsed['template']}[{parsed['activity_a']}"
            if parsed['activity_b']:
                label += f", {parsed['activity_b']}"
            label += "]"
            print(f"    {int(count):4d}×  {label}")

Top violated constraints per role (full model):

  BOT:
     326×  Response[created, closed]
     326×  Responded Existence[created, closed]
     326×  Alternate Response[created, closed]

  CONTRIBUTOR: no violations

  DEVOPS_ENGINEER: no violations

  FEATURE_DEVELOPER: no violations

  ISSUE_REPORTER:
     349×  Alternate Precedence[created, closed]
     326×  Responded Existence[closed, created]
     326×  Precedence[created, closed]
       8×  Not Chain Precedence[created, closed]
       8×  Not Chain Response[created, closed]

  MAINTAINER: no violations

  QUALITY_ENGINEER: no violations

  TECHNICAL_WRITER: no violations
